In [ ]:
import warnings
warnings.filterwarnings('ignore')
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import chromadb
import os
from groq import Groq

In [ ]:
GROQ_API_KEY="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"
os.environ['GROQ_API_KEY']=GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

In [ ]:
df=pd.read_csv('/content/college_notes (1).csv')
print("Shape of dataset :",df.shape)

print('Column names',df.columns.tolist())

Shape of dataset : (15, 4)
Column names ['note_id', 'subject', 'topic', 'content']


In [ ]:
df.head()

,note_id,subject,topic,content
0,1,Data Engineering,ETL Pipelines,"ETL stands for Extract, Transform, Load. It is..."
1,2,Data Engineering,Data Warehousing,A data warehouse is a central repository that ...
2,3,Data Engineering,Apache Spark,Apache Spark is an open-source distributed com...
3,4,Data Engineering,Medallion Architecture,Medallion Architecture is a data design patter...
4,5,Data Engineering,Data Pipelines,A data pipeline is a series of automated steps...


In [ ]:
print("Subjects in the datset")
print(df['subject'].value_counts())

print("\n Sample of topics :")
print(df[['note_id','subject','topic']].to_string(index=False))

print("Length of content for each notes (no of char)")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']])

Subjects in the datset
subject
Data Engineering    5
GenAI               5
Machine Learning    3
Python              2
Name: count, dtype: int64

 Sample of topics :
 note_id          subject                  topic
       1 Data Engineering          ETL Pipelines
       2 Data Engineering       Data Warehousing
       3 Data Engineering           Apache Spark
       4 Data Engineering Medallion Architecture
       5 Data Engineering         Data Pipelines
       6 Machine Learning      Linear Regression
       7 Machine Learning    Feature Engineering
       8 Machine Learning       Model Evaluation
       9            GenAI  Large Language Models
      10            GenAI     Prompt Engineering
      11            GenAI             Embeddings
      12            GenAI       Vector Databases
      13            GenAI            RAG Systems
      14           Python         Pandas Library
      15           Python        API Integration
Length of content for each notes (no of char)
    

CHUNKING

In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas=[
    {"subject": row['subject'],"topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared :{len(documents)}")
print(f"First Document ID : {ids[0]}")
print(f"First Document Metadata : {metadatas[0]}")
print(f"First 100 chars of doc : {documents[0][:100]}...")

Total chunks prepared :15
First Document ID : note_1
First Document Metadata : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc : ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from ...


In [ ]:
print("Loading embedding model...")
print("This may take 30-60 seconds on first run")
embeddding_model=SentenceTransformer('all-MiniLM-L6-v2')
print(" Loaded")

Loading embedding model...
This may take 30-60 seconds on first run


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Loaded


In [ ]:
print("Initializing ChromaDB client and collection...")
client = chromadb.Client()
collection = client.get_or_create_collection(name="my_notes_collection")

# Generate embeddings for the documents and add them to the collection
embeddings = embeddding_model.encode(documents).tolist()
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)
print("ChromaDB collection initialized and documents added successfully.")
print(f"Number of items in collection: {collection.count()}")

Initializing ChromaDB client and collection...
ChromaDB collection initialized and documents added successfully.
Number of items in collection: 15


In [ ]:
def retrieve_relevant_chunks(question,top_k=3):
  """
  Given a user question, retrieve the most relevant document chunks from chromaDB,
  Parameters :
       question(str) : The user's question as text string
       top_k(int) : How many top results to return(default:3)
  Returns:
       A dictionary containing retrieved documents, distances andn meta data
  """


In [ ]:
def retrieve_relavent_chunks(question,top_k=3):
  question_embedding=embeddding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=question_embedding,
      n_results=top_k,
  )
  return results
print("Retrieval function defined successfully.")
print("Function: retrieve_relavent_chunks(question,top_k=3)")

Retrieval function defined successfully.
Function: retrieve_relavent_chunks(question,top_k=3)


In [ ]:
test_question="What is ETL and how does it work in data engineering?"
print(f"Test Quetion: {test_question}")
print("="*60)
results=retrieve_relavent_chunks(test_question,top_k=3)
print("\nTop 3 relavent chunks")
print("="*60)
for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"\nResult {i+1}:")
  print(f"  Subject  : {meta['subject']}")
  print(f"  Topic    : {meta['topic']}")
  print(f"  Distance : {dist:.2f}")
  print(f"  Document : {doc}")

Test Quetion: What is ETL and how does it work in data engineering?

Top 3 relavent chunks

Result 1:
  Subject  : Data Engineering
  Topic    : ETL Pipelines
  Distance : 0.31
  Document : ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from source systems into a data warehouse. First, data is extracted from multiple sources such as databases, APIs, or files. Then it is transformed by cleaning errors, converting formats, and applying business rules. Finally, the cleaned data is loaded into a destination system such as a data warehouse or data lake. ETL pipelines are automated to run on a schedule, ensuring fresh data is always available for analysis.

Result 2:
  Subject  : Data Engineering
  Topic    : Data Pipelines
  Distance : 1.29
  Document : A data pipeline is a series of automated steps that move and transform data from one system to another. Pipelines are essential in modern data engineering for ensuring data flows reliably from 

In [ ]:
question_3="What is ETL and what are its three main stages?"
answer_3=ask_college_assistant(question_1, top_k=3,verbose=True)

User question: What is ETL and what are its three main stages?
Retrieving top 3 relevant chunks...

--- Retrieved Context ---
Document 1 (Subject: Data Engineering, Topic: ETL Pipelines, Distance: 0.58):
ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from source systems into a data warehouse. First, data is extracted from multiple sources such as databases, APIs, or files. Then it is transformed by cleaning errors, converting formats, and applying business rules. Finally, the cleaned data is loaded into a destination system such as a data warehouse or data lake. ETL pipelines are automated to run on a schedule, ensuring fresh data is always available for analysis.

Document 2 (Subject: GenAI, Topic: Prompt Engineering, Distance: 1.48):
Prompt engineering is the practice of designing and refining the instructions given to a large language model to get the best possible response. A good prompt is clear, specific, and provides necessary cont

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-quickchat` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
def ask_college_assistant(question,top_k=3,verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("="*60)
    print("Step 1: Retriving relavant documents...")
  results=retrieve_relavent_chunks(question,top_k=top_k)

  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base:")
    for i,meta in enumerate(results['metadatas'][0]):
      print(f" {i+1}. {meta['subject']}-{meta['topic']}")
    print("\nStep 2: building context string...")
  context=build_context_from_results(results)
  if verbose:
    print(f"Context built ({len(context)} characters)")
    print("\nStep 3: Sending to LLM for answer generation...")
  answer=generate_rag_answer(question,context)
  if verbose:
    print("\n"+"="*60)
    print("ANSWER:")
    print("="*60)
    print(answer)
    print("="*60)
  return answer
print("Complete RAG pipeline function ready")
print("Function: ask_college_assistant(question,top_k=3)")

Complete RAG pipeline function ready
Function: ask_college_assistant(question,top_k=3)
